# Plot pcFVA Results

In [ ]:
from pathlib import Path
from itertools import chain
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
from rbc_gem_utils import read_cobra_model

In [ ]:
model_id = "RBC3P_expanded"
dataset_name = "RBComics_G6PD"
data_path = Path("../../../../data/analysis/OVERLAY").resolve()
root_path = Path("../../../..").resolve()
results_path = root_path / "data" / "processed" / model_id / "OVERLAY"
pcmodel_dirpath = data_path / model_id
dataset_path = results_path / dataset_name
dataset_models_dirpath = dataset_path / "pcmodels"
pcfva_results_dirpath = dataset_path / "pcFVA"
corr_results_dirpath = dataset_path / "correlations"
df_pcfva_all_filename = pcfva_results_dirpath / f"{model_id}_PC_FVAresults_ALL.tsv"
model_filename = pcmodel_dirpath / f"{model_id}.xml"
pcmodel_filename = pcmodel_dirpath / f"{model_id}_PC.xml"
correlations2_dirpath = dataset_path / "correlations2"
flux_plots_path = dataset_path / "flux_plots"

print(results_path)
print(pcmodel_dirpath)
print(dataset_path)
print(dataset_models_dirpath)
print(pcfva_results_dirpath)
print(corr_results_dirpath)
print(df_pcfva_all_filename)
print(model_filename)
print(pcmodel_filename)
print(correlations2_dirpath)
print(flux_plots_path)

In [ ]:
df_flux_abundance_correlation_filename = (
    correlations2_dirpath / "df_flux_abundance_correlation.csv"
)
df_flux_abundance_correlation = pd.read_csv(df_flux_abundance_correlation_filename)
df_flux_abundance_correlation

In [ ]:
def plot_correlations(
    df, vertical_lines, ax=None, histx=True, histy=True, colorbar=True, **kwargs
):
    # Define figure if no axes provided.
    scatter_inch = kwargs.get("scatter_inch", 5.0)
    hist_inch = kwargs.get("hist_inch", 1.0)
    hist_pad = kwargs.get("hist_pad", 0.25)
    if ax is None:
        _, ax = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(
                scatter_inch + (hist_inch + hist_pad if histy else 0),
                scatter_inch + (hist_inch + hist_pad if histx else 0),
            ),
        )
    xy = {"x": "rho", "y": "neg_log10_adj_p_value"}
    limits = {
        "x": (kwargs.get("xmin", -1.0), kwargs.get("xmax", 1.0)),
        "y": (kwargs.get("ymin", 0.0), kwargs.get("ymax", df[xy["y"]].max())),
    }
    pads = {
        axis: kwargs.get(f"{axis}pad", (limits[axis][1] - limits[axis][0]) / 2 / 20)
        for axis in list(xy)
    }
    cmap = kwargs.get("cmap", "viridis")
    zorder = kwargs.get("zorder", 2)
    edgecolor = kwargs.get("edgecolor", "black")
    edgewidth = kwargs.get("edgewidth", 0.5)
    scatter = ax.scatter(
        xy["x"],
        xy["y"],
        data=df,
        c=kwargs.get("c", xy["y"]),
        s=kwargs.get("s", 40),
        alpha=0.5,
        zorder=zorder,
        edgecolor=edgecolor,
        linewidth=edgewidth,
        cmap=mpl.colormaps.get_cmap(cmap) if isinstance(cmap, str) else cmap,
        norm=mpl.colors.Normalize(
            vmin=limits["y"][0] - pads["y"], vmax=limits["y"][1] + pads["y"]
        ),
    )
    ax.set_xlabel(r"Spearman Correlation $(\rho)$", fontdict={"size": "xx-large"})
    ax.set_ylabel("-log$_{10}$(adj p-value)", fontdict={"size": "xx-large"})
    ax.set_xlim((limits["x"][0] - pads["x"], limits["x"][1] + pads["x"]))
    ax.set_ylim((limits["y"][0] - pads["y"], limits["y"][1] + pads["y"]))

    major_ticks = {axis: kwargs.get(f"{axis}tick_major") for axis in list(xy)}
    minor_ticks = {
        axis: kwargs.get(
            f"{axis}tick_minor",
            major_ticks[axis] / 2 if major_ticks[axis] is not None else None,
        )
        for axis in list(xy)
    }
    for axis in list(xy):
        if major_ticks[axis] is not None:
            getattr(ax, f"{axis}axis").set_major_locator(
                mpl.ticker.MultipleLocator(major_ticks[axis])
            )
        if minor_ticks[axis] is not None:
            getattr(ax, f"{axis}axis").set_minor_locator(
                mpl.ticker.MultipleLocator(minor_ticks[axis])
            )
        ax.tick_params(axis=axis, labelsize="large")

    if vertical_lines:
        for lineval, (lineprops, textprops) in vertical_lines.items():
            if lineprops:
                ax.vlines(
                    x=lineval,
                    ymin=limits["y"][0] - pads["y"],
                    ymax=limits["y"][1] + pads["y"],
                    **lineprops,
                )
            if textprops:
                ax.text(x=lineval + pads["x"] / 2, transform=ax.transData, **textprops)

    if kwargs.get("grid", False):
        ax.grid(True, **dict(which="both", alpha=0.75))

    if colorbar:
        cax = ax.inset_axes(
            [
                limits["x"][0] - pads["x"],  # lower left corner xpos
                limits["y"][0] - pads["y"],  # lower left corner ypos
                pads["x"],  # width of colorbar
                limits["y"][1]
                + pads["y"]
                + pads[
                    "y"
                ],  # height of colorbar, need extra ypad to make up for lowering ypos
            ],
            transform=ax.transData,
        )
        cbar = ax.get_figure().colorbar(scatter, cax=cax)
        cax.set_ylim((limits["y"][0] - pads["y"], limits["y"][1] + pads["y"]))
        cax.set_xticks([])
        cax.set_yticks([])

    ax_histx = None
    ax_histy = None
    if histx or histy:
        divider = make_axes_locatable(ax)
        # Histogram axes
        ax_histx = (
            divider.append_axes("top", hist_inch, pad=hist_pad, sharex=ax)
            if histx
            else None
        )
        ax_histy = (
            divider.append_axes("right", hist_inch, pad=hist_pad, sharey=ax)
            if histy
            else None
        )

        for axis, ax_hist in zip(list(xy), [ax_histx, ax_histy]):
            if ax_hist is None:
                continue
            binwidth = kwargs.get(
                f"{axis}binwidth",
                (
                    minor_ticks[axis]
                    if minor_ticks[axis] is not None
                    else major_ticks[axis]
                ),
            )
            counts, bins, patches = ax_hist.hist(
                df[xy[axis]],
                bins=np.arange(limits[axis][0], limits[axis][1] + binwidth, binwidth),
                orientation="vertical" if axis == "x" else "horizontal",
                zorder=zorder,
                edgecolor=edgecolor,
                linewidth=edgewidth,
            )
            other = "y" if axis == "x" else "x"
            ax_hist.tick_params(
                axis=axis, **{f"label{'bottom' if axis == 'x' else 'left'}": False}
            )
            ax_hist.tick_params(axis=other, labelsize="large")
            getattr(ax_hist, f"set_{other}label")("Frequency", fontsize="large")

            tick_major_int = kwargs.get(f"hist{axis}_{other}tick_major")
            if tick_major_int is not None:
                getattr(ax_hist, f"{other}axis").set_major_locator(
                    mpl.ticker.MultipleLocator(tick_major_int)
                )
                getattr(ax_hist, f"{other}axis").set_minor_locator(
                    mpl.ticker.MultipleLocator(tick_major_int / 2)
                )
            getattr(ax_hist, f"set_{other}lim")((0, max(counts) * 1.1))
            if kwargs.get("grid", False):
                ax_hist.grid(True, **dict(which="both", alpha=0.75))

            if vertical_lines and (axis == "x" and ax_hist is not None):
                for lineval, (lineprops, _) in vertical_lines.items():
                    if lineprops:
                        ax_hist.vlines(
                            x=lineval, ymin=0.0, ymax=max(counts) * 1.1, **lineprops
                        )

    return ax, ax_histx, ax_histy

In [ ]:
def make_flux_abundance_u_plot(df, histx=True, histy=True, colorbar=True, **kwargs):
    scatter_inch = kwargs.get("scatter_inch", 5.0)
    hist_inch = kwargs.get("hist_inch", 1.0)
    hist_pad = kwargs.get("hist_pad", 0.5)
    nrows, ncols = (1, 1)
    expression_dep_rho_lb = 0.8
    expression_cor_rho_lb = 0.5
    ypos = 4
    ww = 11
    rotation = 90
    fontsize = "large"
    linewidth = 2
    vertical_lines = {
        expression_dep_rho_lb: (
            dict(color="black", linestyle="-", linewidth=linewidth),
            dict(
                y=ypos,
                s="\n".join(textwrap.wrap("Expression dependent", width=ww)),
                rotation=rotation,
                fontsize=fontsize,
            ),
        ),
        expression_cor_rho_lb: (
            dict(color="xkcd:dark grey", linestyle="--", linewidth=linewidth),
            dict(
                y=ypos,
                s="\n".join(textwrap.wrap("Expression correlated", width=ww)),
                rotation=rotation,
                fontsize=fontsize,
            ),
        ),
        0.0: (
            dict(),
            dict(
                y=ypos + 50.0,
                s="\n".join(textwrap.wrap("Expression independent", width=ww)),
                rotation=rotation,
                fontsize=fontsize,
            ),
        ),
    }
    fig, ax = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(
            (scatter_inch + (hist_inch + hist_pad if histx else 0)) * ncols,
            (scatter_inch + (hist_inch + hist_pad if histy else 0)) * nrows,
        ),
    )
    ax_scatter, ax_histx, ax_histy = plot_correlations(
        df,
        ax=ax,
        histx=histx,
        histy=histy,
        colorbar=True,
        vertical_lines=vertical_lines,
        xbinwidth=0.1,
        ybinwidth=10,
        **kwargs,
    )
    # ax_scatter.set_title(
    #     f"Correlates between Flux and Abundance",
    #     fontsize="x-large",
    # )
    fig.suptitle("Correlates Between Flux and Abundance", fontsize=18)

In [ ]:
make_flux_abundance_u_plot(df_flux_abundance_correlation)
correlates_flux_abundance_plot_filename = (
    flux_plots_path / "correlates_flux_abundance.svg"
)
plt.savefig(
    correlates_flux_abundance_plot_filename,
    dpi=300,
    transparent=False,
    bbox_inches="tight",
    pad_inches=0.5,
    format="svg",
)
plt.close()

### Donut charts of subsystems

Look at categories and subsystems here: [KEGG pathways](https://www.kegg.jp/kegg/pathway.html) Look at "category" in df_pathways.

In [ ]:
df_flux_abundance_correlation["category"].unique()

In [ ]:
def map_categories_to_colors(categories):
    color_for_category = {
        "Nucleotide metabolism": "#332288",
        "Transport reactions": "#117733",
        "Metabolism of cofactors and vitamins": "#44AA99",
        "Carbohydrate metabolism": "#88CCEE",
        "Reactive species": "#999933",
        "Metabolism of other amino acids": "#DDCC77",
    }
    return [color_for_category[category] for category in categories]

In [ ]:
def make_subsystem_breakdown_by_abundance_dependence(
    df, abundance_dependence, figsize=(8, 8), title_y=1.2, center_title=None
):
    color_for_category = {
        "Nucleotide metabolism": "#332288",
        "Transport reactions": "#117733",
        "Metabolism of cofactors and vitamins": "#44AA99",
        "Carbohydrate metabolism": "#88CCEE",
        "Reactive species": "#999933",
        "Metabolism of other amino acids": "#DDCC77",
    }
    ring_series = df[df["abundance_dependence"] == abundance_dependence]["category"]
    ring_series = ring_series.value_counts()
    wedge_labels = [
        f"{value}: {'\n'.join(textwrap.wrap(index, width=20))}"
        for index, value in ring_series.items()
    ]
    wedge_colors = [color_for_category[category] for category, _ in ring_series.items()]
    # Create a figure and axis
    fig, ax = plt.subplots(figsize=figsize)
    wedges, texts = ax.pie(
        ring_series,
        labels=wedge_labels,
        startangle=90,
        wedgeprops={"edgecolor": "white"},
        colors=wedge_colors,
        labeldistance=1.2,
    )
    center_circle = plt.Circle((0, 0), 0.75, fc="white")
    ax.add_artist(center_circle)
    ax.set_aspect("equal")
    formatted_center_title = (
        f"{center_title}\n{sum(ring_series)}" if center_title else sum(ring_series)
    )
    ax.text(0, 0, formatted_center_title, ha="center", va="center", fontsize=12)
    plt.title(abundance_dependence, fontsize=14, y=title_y)

In [ ]:
make_subsystem_breakdown_by_abundance_dependence(
    df_flux_abundance_correlation,
    abundance_dependence="Dependent",
    figsize=(3, 3),
    center_title="ρ ≥ 0.8",
)
dependent_reaction_categories_plot_filename = (
    flux_plots_path / "dependent_reaction_categories.svg"
)
plt.savefig(
    dependent_reaction_categories_plot_filename,
    dpi=300,
    transparent=False,
    bbox_inches="tight",
    pad_inches=0.5,
    format="svg",
)

In [ ]:
make_subsystem_breakdown_by_abundance_dependence(
    df_flux_abundance_correlation,
    abundance_dependence="Correlated",
    figsize=(3, 3),
    center_title="0.5 ≤ ρ < 0.8",
)
correlated_reaction_categories_plot_filename = (
    flux_plots_path / "correlated_reaction_categories.svg"
)
plt.savefig(
    correlated_reaction_categories_plot_filename,
    dpi=300,
    transparent=False,
    bbox_inches="tight",
    pad_inches=0.5,
    format="svg",
)

In [ ]:
make_subsystem_breakdown_by_abundance_dependence(
    df_flux_abundance_correlation,
    abundance_dependence="Independent",
    figsize=(3, 3),
    center_title="ρ < 0.5",
)
independent_reaction_categories_plot_filename = (
    flux_plots_path / "independent_reaction_categories.svg"
)
plt.savefig(
    independent_reaction_categories_plot_filename,
    dpi=300,
    transparent=False,
    bbox_inches="tight",
    pad_inches=0.5,
    format="svg",
)

TODO: Make pathway map. Use Escher and instead of edge weights being fluxes, correlation coefficients.

### Gene reaction counts horizontal bar charts

In [ ]:
df_gene_reaction_count_filename = correlations2_dirpath / "df_gene_reaction_count.csv"
df_gene_reaction_count = pd.read_csv(df_gene_reaction_count_filename)
df_gene_reaction_count

In [ ]:
def make_gene_reaction_count_plot(df, abundance_dependence, **kwargs):
    df1 = df[df["abundance_dependence"] == abundance_dependence][
        ["reactions", "genes", "category"]
    ]
    df1.set_index("genes", inplace=True)
    fig, ax = plt.subplots(nrows=1, ncols=1, **kwargs)
    ax.barh(
        df1.index, df1["reactions"], color=map_categories_to_colors(df1["category"])
    )

In [ ]:
make_gene_reaction_count_plot(df_gene_reaction_count, "Dependent", figsize=(5, 11))

In [ ]:
make_gene_reaction_count_plot(df_gene_reaction_count, "Correlated", figsize=(5, 2))

In [ ]:
make_gene_reaction_count_plot(df_gene_reaction_count, "Independent", figsize=(5, 18))

### Plot solution space of by day, reaction, and G6PD_allele

In [ ]:
df_pcfva_alleles_filename = correlations2_dirpath / "df_pcfva_alleles.csv"
df_pcfva_alleles = pd.read_csv(df_pcfva_alleles_filename)
df_pcfva_alleles.drop("sample_id", axis=1, inplace=True)
df_pcfva_alleles.set_index(
    ["G6PD_alleles", "day", "reactions", "optimum"], inplace=True
)
df_pcfva_alleles.sort_index(inplace=True)
df_pcfva_alleles

In [ ]:
model = read_cobra_model(model_filename)
subsystem_reaction = {}
for reaction in model.reactions:
    subsystem = reaction.subsystem
    if subsystem in subsystem_reaction:
        subsystem_reaction[subsystem].append(reaction.id)
    else:
        subsystem_reaction[subsystem] = [reaction.id]
for subsystem, reaction_list in subsystem_reaction.items():
    joined_reaction_list = ", ".join(reaction_list)
    print(subsystem, len(reaction_list), joined_reaction_list)

In [ ]:
flux_group_01 = [
    ["HEX1", "PGI", "PFK", "TPI", "FBA"],  # Upper glycolysis
    ["GAPD", "PGK", "PGM", "ENO", "PYK"],  # Lower glycolysis
    ["G6PDH2", "GND", "PGL", "PPM", "PRPPS"],  # Partial PPP
    ["BILIREDy", "FE2O2OX", "FE3RD", "FE2H2O2X", "SPODM"],  # Reactive species formation and detoxification, Porphyrin metabolism
    ["LDH_L", "DPGM", "DPGase", "HB23DPGB", "NaKt"],  # Hemoglobin and objective
]
fluxes_to_plot_np = np.array(flux_group_01)

flux_group_ppp = [
    ["G6PDH2", "GND", "PGL", "PPM", "PRPPS"],
    ["RPE", "RPI", "TALA", "TKT1", "TKT2"]
]

flux_group_glycolysis = [
    ["DPGM", "DPGase", "ENO", "FBA"],
    ["HEX1", "GAPD", "LDH_L", "PFK"],
    ["PGI", "PGK", "PGM", "PYK"]  # Skipped TPI
]

flux_group_purine_metabolism = [
    ["ADA", "ADK1", "ADNK1"],
    ["ADPT", "AMPDA", "HXPRT"],
    ["NTDIMP", "NTDAMP", "PUNP5"]
]

In [ ]:
def make_optimum_min_max_plot(
    day, reaction, optima=None, optimum_colors=None, save_filename=None, **kwargs
):
    optima = [0.0, 0.5, 0.9, 0.99] if not optima else optima
    optimum_colors = (
        ["#87CEEB", "#3399CC", "#004C99", "#000080"]
        if not optimum_colors
        else optimum_colors
    )
    fig, axs = plt.subplots(nrows=1, ncols=3, sharey=True, **kwargs)
    for allele_count, (ax_idx, ax) in zip(range(3), enumerate(axs)):
        for optimum, optimum_color in zip(optima, optimum_colors):
            multi = (allele_count, day, reaction, optimum)
            df_for_plot = df_pcfva_alleles.loc[multi, :].sort_values(by="range").copy()
            y_mins = df_for_plot["minimum"]
            y_maxs = df_for_plot["maximum"]
            xs = np.arange(1, len(y_maxs) + 1)
            ax.fill_between(
                xs,
                y_mins,
                y_maxs,
                color=optimum_color,
                label=f"{optimum*100:.0f}% Max NaKt",
            )
            ax.set_xlabel(allele_count, fontsize=14)
            ax.set_xticks([])
            if ax_idx == 0:
                ax.set_ylabel("Flux (mmol/gDW/hr)", fontsize=14)
            if ax_idx == 2:
                ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    fig.suptitle(f"{reaction}, Day {day}", fontsize=18)
    if save_filename:
        plt.savefig(
            save_filename,
            dpi=300,
            transparent=False,
            bbox_inches="tight",
            pad_inches=0.5,
            format="svg",
        )
    plt.close()

In [ ]:
def iterate_make_optimum_min_max_plots(fluxes_to_plot):
    flat_list = list(chain(*fluxes_to_plot))
    for flux in flat_list:
        for day in [10, 23, 42]:
            save_filename = flux_plots_path / f"{flux}_day_{day}.svg"
            make_optimum_min_max_plot(
                day, flux, figsize=(5, 2.5), save_filename=save_filename
            )
            print(save_filename, "saved")

In [ ]:
iterate_make_optimum_min_max_plots(flux_group_01)

### Combine these plots onto one plot with small multiples

In [ ]:
def make_flux_alleles_positions_for_day(day, fluxes_to_plot):
    flux_alleles_positions = []
    for row_idx, row in enumerate(fluxes_to_plot):
        flux_alleles_positions.append([])
        for col_idx, reaction in enumerate(row):
            day_reaction_alleles = [
                [0, day, reaction],
                [1, day, reaction],
                [2, day, reaction],
            ]
            flux_alleles_positions[row_idx].append(day_reaction_alleles)
    return flux_alleles_positions

In [ ]:
def alleles_day_flux_small_multiples(flux_alleles_positions, title, figsize=(15, 8)):
    fig = plt.figure(figsize=figsize)
    zero_one_two_alleles = 3
    rows = len(flux_alleles_positions)
    cols = len(flux_alleles_positions[0]) * zero_one_two_alleles
    optima = [0.0, 0.5, 0.9, 0.99]
    optima_colors = ["#87CEEB", "#3399CC", "#004C99", "#000080"]
    ylim_max = 0.2
    ylim_min = -0.2
    # TODO: make_axes_locatable?
    gs = GridSpec(rows, cols, wspace=0)
    for gs_row, day_flux_alleles_positions_row in zip(
        range(rows), flux_alleles_positions
    ):
        for gs_col_left, alleles_day_flux_group in zip(
            range(0, cols, zero_one_two_alleles), day_flux_alleles_positions_row
        ):
            for group_idx in range(zero_one_two_alleles):
                alleles, day, flux = alleles_day_flux_group[group_idx]
                ax = fig.add_subplot(gs[gs_row, gs_col_left + group_idx])
                ax.set_ylim(ylim_min, ylim_max)
                if group_idx == 0:
                    ax.tick_params(right=False)
                    ax.spines["right"].set_visible(False)
                    ax.set_title(flux)
                elif group_idx == 1:
                    ax.tick_params(left=False, right=False)
                    ax.spines["left"].set_visible(False)
                    ax.spines["right"].set_visible(False)
                    ax.set_yticklabels([])
                else:
                    ax.spines["left"].set_visible(False)
                    ax.tick_params(left=False)
                    ax.set_yticklabels([])
                if gs_col_left > 0:
                    ax.set_yticklabels([])
                    ax.tick_params(left=False)
                for optimum, optimum_color in zip(optima, optima_colors):
                    multi_index = (alleles, day, flux, optimum)
                    df = df_pcfva_alleles.loc[multi_index].sort_values(by="range").copy()
                    y_mins = df["minimum"]
                    y_maxs = df["maximum"]
                    xs = np.arange(1, len(y_maxs) + 1)
                    ax.fill_between(
                        xs,
                        y_mins,
                        y_maxs,
                        color=optimum_color,
                        label=f"{optimum*100:.0f}% Max NaKt",
                    )
                    ax.set_xlabel(alleles, fontsize=14)
                    ax.set_xticks([])
    fig.suptitle(f"{title} Storage Day {day}", fontsize=24)
    fig.tight_layout()
    clean_title = title.lower().strip().replace(" ", "_")
    flux_small_mutiples_plot_filename = (
        flux_plots_path / f"{clean_title}_small_mutiples_day_{day}.svg"
    )
    plt.savefig(
        flux_small_mutiples_plot_filename,
        dpi=300,
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.5,
        format="svg",
    )

In [ ]:
# flux_alleles_positions_10 = make_flux_alleles_positions_for_day(10, flux_group_01)
# alleles_day_flux_small_multiples(flux_alleles_positions_10, "Flux Group 01")

In [ ]:
flux_group_ppp_positions_10 = make_flux_alleles_positions_for_day(10, flux_group_ppp)
alleles_day_flux_small_multiples(flux_group_ppp_positions_10, "PPP", figsize=(15, 4))

In [ ]:
flux_group_glycolysis_positions_10 = make_flux_alleles_positions_for_day(10, flux_group_glycolysis)
alleles_day_flux_small_multiples(flux_group_glycolysis_positions_10, "Glycolysis", figsize=(15, 7))